# Phase 1: Extraction of Liver Cancer Cell Line Data

Load the liver cancer cell line list and retrieve the corresponding cell line data directly from Snowflake.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging
from typing import List, Optional
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect
from scipy import stats

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

## Snowflake Connection Settings

Define functions required for connecting to the Snowflake database. Establish connection using private key authentication.

In [ ]:
def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    # Encode in DER format (format expected by Snowflake connector)
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user="KOREEDA",
            account="DUETMBM-LL33279",
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

## Loading Liver Cancer Cell Line List

Load the pre-created liver cancer cell line list (CSV file) and identify the target cell lines for data extraction.

In [ ]:
# Load liver cancer cell line list https://lincsportal.ccs.miami.edu/cells/#/catalog
liver_cells_path = project_root / "results" / "liver_cell_lines_list.csv"
df_liver_list = pd.read_csv(liver_cells_path)
cell_lines = df_liver_list['CELL_LINE_NAME'].tolist()

print(f"Loaded liver cancer cell line list: {len(cell_lines)} types")
print(f"Cell lines: {', '.join(cell_lines)}")
df_liver_list

## Connecting to Snowflake

Connect to the Snowflake database and prepare to retrieve data from the GLYCO_GENES_WIDE table.

In [ ]:
conn = connect_to_snowflake()


## Retrieving Glycogene Columns

Retrieve column information from the GLYCO_GENES_WIDE table, exclude metadata columns, and create a list of glycogene columns.

In [ ]:
# Get glycogene column names from GLYCO_GENES_WIDE table
metadata_columns = {
    'VALUE', 'canonical_smiles', 'cell', 'cmapid', 'compound_alias', 
    'dose', 'inchi_key', 'pertid', 'pertname', 'timepoint', 'sample_id'
}

column_query = """
SELECT COLUMN_NAME 
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_SCHEMA = 'LINCS' 
AND TABLE_NAME = 'GLYCO_GENES_WIDE' 
ORDER BY COLUMN_NAME
"""

column_df = pd.read_sql(column_query, conn)
all_columns = column_df['COLUMN_NAME'].tolist()
glyco_genes = [col for col in all_columns if col not in metadata_columns]

print(f"Number of glycogene columns: {len(glyco_genes)}")
print(f"First 10: {glyco_genes[:10]}")

## Gene Coverage Comparison between GlycoEnzOnto and LINCS L1000

Identify genes in the GlycoEnzOnto ontology that are not included in the LINCS L1000 dataset (GLYCO_GENES_WIDE table).

In [ ]:
# Extract all glycogenes from GlycoEnzOnto GMT file
gmt_path = project_root / 'GlycoEnzOnto' / 'GlycoEnzOnto.gmt'

glycoenzonto_all = set()
pathway_gene_map = {}  # Also maintain pathway to gene mapping
with open(gmt_path) as f:
    for line in f:
        parts = line.strip().split('\t')
        pathway_name = parts[0].strip('"')
        genes = [g.strip('"') for g in parts[2:]]
        pathway_gene_map[pathway_name] = genes
        glycoenzonto_all.update(genes)

print(f'Total GlycoEnzOnto genes: {len(glycoenzonto_all)}')
print(f'Genes in LINCS L1000: {len(glyco_genes)}')

# Genes not included in LINCS L1000
lincs_set = set(glyco_genes)
missing_genes = sorted(glycoenzonto_all - lincs_set)
covered_genes = sorted(glycoenzonto_all & lincs_set)

print(f'\nGenes not in LINCS L1000: {len(missing_genes)}')
print(f'Coverage: {len(covered_genes)}/{len(glycoenzonto_all)} ({100*len(covered_genes)/len(glycoenzonto_all):.1f}%)')
print(f'\n--- List of missing genes ---')
for g in missing_genes:
    # Show which pathway they belong to
    pathways = [pw for pw, genes in pathway_gene_map.items() if g in genes]
    print(f'  {g:20s} <- {pathways[0] if pathways else "N/A"}')

# Compile into DataFrame and save
df_coverage = pd.DataFrame({
    'gene': sorted(glycoenzonto_all),
    'in_lincs': [g in lincs_set for g in sorted(glycoenzonto_all)]
})
df_coverage['pathways'] = df_coverage['gene'].apply(
    lambda g: '; '.join([pw for pw, genes in pathway_gene_map.items() if g in genes])
)

results_dir = project_root / 'results' / 'data_qc'
results_dir.mkdir(parents=True, exist_ok=True)
df_coverage.to_csv(results_dir / 'glycoenzonto_lincs_coverage.csv', index=False)
print(f'\nSaved: {results_dir / "glycoenzonto_lincs_coverage.csv"}')

## Gene Coverage Comparison between GSE92742 Table and GlycoEnzOnto

Retrieve glycogenes from the GLYCO_GENES_WIDE_GSE92742 table and check differences with GlycoEnzOnto and the GLYCO_GENES_WIDE table used in this analysis.

In [ ]:
# Get glycogene columns from GSE92742 table
gse92742_metadata_columns = {'SAMPLE_ID', 'PERTID', 'CELL', 'DOSE', 'TIMEPOINT'}

gse92742_col_query = """
SELECT COLUMN_NAME 
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_SCHEMA = 'LINCS' 
AND TABLE_NAME = 'GLYCO_GENES_WIDE_GSE92742' 
ORDER BY COLUMN_NAME
"""

gse92742_col_df = pd.read_sql(gse92742_col_query, conn)
gse92742_all_cols = gse92742_col_df['COLUMN_NAME'].tolist()
gse92742_genes = set(col for col in gse92742_all_cols if col not in gse92742_metadata_columns)

print(f'GSE92742 table total columns: {len(gse92742_all_cols)}')
print(f'GSE92742 table metadata columns: {sorted(gse92742_metadata_columns & set(gse92742_all_cols))}')
print(f'GSE92742 table glycogene count: {len(gse92742_genes)}')
print(f'GLYCO_GENES_WIDE glycogene count: {len(glyco_genes)}')
print(f'Total GlycoEnzOnto genes:      {len(glycoenzonto_all)}')

# --- GlycoEnzOnto vs GSE92742 ---
missing_from_gse92742 = sorted(glycoenzonto_all - gse92742_genes)
covered_gse92742 = glycoenzonto_all & gse92742_genes
print(f'\n=== GlycoEnzOnto vs GSE92742 ===')
print(f'Coverage: {len(covered_gse92742)}/{len(glycoenzonto_all)} ({100*len(covered_gse92742)/len(glycoenzonto_all):.1f}%)')
print(f'Genes not in GSE92742: {len(missing_from_gse92742)}')
for g in missing_from_gse92742:
    pathways = [pw for pw, genes in pathway_gene_map.items() if g in genes]
    print(f'  {g:20s} <- {pathways[0] if pathways else "N/A"}')

# --- Differences between the two tables ---
lincs_set = set(glyco_genes)
only_in_wide = sorted(lincs_set - gse92742_genes)
only_in_gse92742 = sorted(gse92742_genes - lincs_set)
common_genes = lincs_set & gse92742_genes
print(f'\n=== GLYCO_GENES_WIDE vs GSE92742 ===')
print(f'Common genes:              {len(common_genes)}')
print(f'CycleGAN only ({len(only_in_wide)}):  {only_in_wide}')
print(f'GSE92742 only ({len(only_in_gse92742)}): {only_in_gse92742}')

# Update and save coverage table
df_coverage['in_gse92742'] = df_coverage['gene'].apply(lambda g: g in gse92742_genes)
df_coverage.to_csv(
    project_root / 'results' / 'data_qc' / 'glycoenzonto_lincs_coverage.csv',
    index=False
)
print(f'\nSaved: glycoenzonto_lincs_coverage.csv (added in_gse92742 column)')

## GSE92742 vs CycleGAN Gene Coverage Report

Summarize the gene availability in two LINCS L1000 data sources (GSE92742 original / CycleGAN predicted RNA-Seq-like profiles) and their coverage of GlycoEnzOnto.

In [ ]:
# ========================================
# GSE92742 vs CycleGAN 遺伝子カバレッジレポート
# ========================================

lincs_set = set(glyco_genes)          # CycleGAN (GLYCO_GENES_WIDE)
# gse92742_genes は前セルで取得済み

common = sorted(lincs_set & gse92742_genes)
only_cyclegan = sorted(lincs_set - gse92742_genes)
only_gse92742 = sorted(gse92742_genes - lincs_set)
all_union = lincs_set | gse92742_genes

# GlycoEnzOntoに対するカバレッジ
cyclegan_covered = glycoenzonto_all & lincs_set
gse92742_covered = glycoenzonto_all & gse92742_genes
union_covered = glycoenzonto_all & all_union
neither = sorted(glycoenzonto_all - all_union)

# --- レポート出力 ---
report = []
report.append('=' * 70)
report.append('GSE92742 vs CycleGAN glycogene カバレッジレポート')
report.append('=' * 70)
report.append('')
report.append('1. 総遺伝子数')
report.append(f'   GlycoEnzOnto全遺伝子:       {len(glycoenzonto_all)}')
report.append(f'   CycleGAN (GLYCO_GENES_WIDE): {len(lincs_set)}')
report.append(f'   GSE92742:                     {len(gse92742_genes)}')
report.append(f'   両テーブル和集合:             {len(all_union)}')
report.append(f'   両テーブル共通:               {len(common)}')
report.append('')
report.append('2. GlycoEnzOntoカバレッジ')
report.append(f'   CycleGAN:   {len(cyclegan_covered)}/{len(glycoenzonto_all)} ({100*len(cyclegan_covered)/len(glycoenzonto_all):.1f}%)')
report.append(f'   GSE92742:   {len(gse92742_covered)}/{len(glycoenzonto_all)} ({100*len(gse92742_covered)/len(glycoenzonto_all):.1f}%)')
report.append(f'   和集合:     {len(union_covered)}/{len(glycoenzonto_all)} ({100*len(union_covered)/len(glycoenzonto_all):.1f}%)')
report.append(f'   どちらにもない: {len(neither)}')
report.append('')
report.append('3. CycleGANのみに含まれる遺伝子 (GSE92742にない)')
if only_cyclegan:
    for g in only_cyclegan:
        pw = [p for p, gs in pathway_gene_map.items() if g in gs]
        report.append(f'   {g:20s} <- {pw[0] if pw else "N/A"}')
else:
    report.append('   (なし)')
report.append('')
report.append('4. GSE92742のみに含まれる遺伝子 (CycleGANにない)')
if only_gse92742:
    for g in only_gse92742:
        pw = [p for p, gs in pathway_gene_map.items() if g in gs]
        report.append(f'   {g:20s} <- {pw[0] if pw else "N/A"}')
else:
    report.append('   (なし)')
report.append('')
report.append('5. GlycoEnzOntoのうちどちらのテーブルにも含まれない遺伝子')
if neither:
    for g in neither:
        pw = [p for p, gs in pathway_gene_map.items() if g in gs]
        report.append(f'   {g:20s} <- {pw[0] if pw else "N/A"}')
else:
    report.append('   (なし)')
report.append('')

# サマリーテーブル
report.append('6. サマリーテーブル')
df_summary = pd.DataFrame({
    'データソース': ['CycleGAN', 'GSE92742', '共通', 'CycleGANのみ', 'GSE92742のみ', '和集合'],
    '総遺伝子数': [len(lincs_set), len(gse92742_genes), len(common), len(only_cyclegan), len(only_gse92742), len(all_union)],
    'GlycoEnzOntoカバー数': [len(cyclegan_covered), len(gse92742_covered), len(glycoenzonto_all & set(common)), '-', '-', len(union_covered)],
    'カバレッジ(%)': [
        f'{100*len(cyclegan_covered)/len(glycoenzonto_all):.1f}',
        f'{100*len(gse92742_covered)/len(glycoenzonto_all):.1f}',
        f'{100*len(glycoenzonto_all & set(common))/len(glycoenzonto_all):.1f}',
        '-', '-',
        f'{100*len(union_covered)/len(glycoenzonto_all):.1f}'
    ]
})

report_text = '\n'.join(report)
print(report_text)
print()
display(df_summary)

# レポート保存
report_dir = project_root / 'results' / 'data_qc'
report_dir.mkdir(parents=True, exist_ok=True)
with open(report_dir / 'gse92742_vs_cyclegan_coverage_report.txt', 'w') as f:
    f.write(report_text)
df_summary.to_csv(report_dir / 'gse92742_vs_cyclegan_coverage_summary.csv', index=False)
print(f'保存: {report_dir / "gse92742_vs_cyclegan_coverage_report.txt"}')
print(f'保存: {report_dir / "gse92742_vs_cyclegan_coverage_summary.csv"}')

## Breakdown of Landmark vs Inferred Genes

Cross-reference with the LINCS L1000 978 landmark gene list to check the breakdown of directly measured (landmark) vs predicted (inferred) genes among the 385 glycogenes used in this analysis.

In [ ]:
# 978 landmark遺伝子リストを読み込み
landmark_path = project_root / '..' / 'data' / 'landmark_genes_978.txt'
with open(landmark_path) as f:
    landmark_genes = set(line.strip() for line in f if line.strip())
print(f'Landmark遺伝子数: {len(landmark_genes)}')

# CycleGAN 385 glycogene との照合
lincs_set = set(glyco_genes)
landmark_glyco = sorted(lincs_set & landmark_genes)
inferred_glyco = sorted(lincs_set - landmark_genes)

print(f'\n=== CycleGAN 385 glycogene の内訳 ===')
print(f'Landmark (直接測定):  {len(landmark_glyco)}/{len(lincs_set)} ({100*len(landmark_glyco)/len(lincs_set):.1f}%)')
print(f'Inferred (推定値):    {len(inferred_glyco)}/{len(lincs_set)} ({100*len(inferred_glyco)/len(lincs_set):.1f}%)')

# GSE92742との照合
# gse92742_genesからメタデータカラムを除外
gse92742_genes_clean = gse92742_genes - {'CELL', 'DOSE', 'PERTID', 'SAMPLE_ID', 'TIMEPOINT'}
landmark_gse92742 = sorted(gse92742_genes_clean & landmark_genes)
inferred_gse92742 = sorted(gse92742_genes_clean - landmark_genes)

print(f'\n=== GSE92742 {len(gse92742_genes_clean)} glycogene の内訳 ===')
print(f'Landmark (直接測定):  {len(landmark_gse92742)}/{len(gse92742_genes_clean)} ({100*len(landmark_gse92742)/len(gse92742_genes_clean):.1f}%)')
print(f'Inferred (推定値):    {len(inferred_gse92742)}/{len(gse92742_genes_clean)} ({100*len(inferred_gse92742)/len(gse92742_genes_clean):.1f}%)')

# Landmark glycogene一覧
print(f'\n=== Landmark glycogene ({len(landmark_glyco)}個) ===')
for i, g in enumerate(landmark_glyco):
    pw = [p for p, gs in pathway_gene_map.items() if g in gs]
    print(f'  {g:20s} <- {pw[0] if pw else "N/A"}')

# サマリーテーブル
df_landmark_summary = pd.DataFrame({
    'データソース': ['CycleGAN (385)', f'GSE92742 ({len(gse92742_genes_clean)})'],
    'Landmark': [len(landmark_glyco), len(landmark_gse92742)],
    'Inferred': [len(inferred_glyco), len(inferred_gse92742)],
    'Landmark比率(%)': [
        f'{100*len(landmark_glyco)/len(lincs_set):.1f}',
        f'{100*len(landmark_gse92742)/len(gse92742_genes_clean):.1f}'
    ]
})
display(df_landmark_summary)

# WGCNAモジュール別のlandmark/inferred内訳（モジュール情報があれば）
# ここではcoverage CSVを更新
df_coverage['is_landmark'] = df_coverage['gene'].apply(lambda g: g in landmark_genes)
df_coverage.to_csv(
    project_root / 'results' / 'data_qc' / 'glycoenzonto_lincs_coverage.csv',
    index=False
)
print(f'\n保存: glycoenzonto_lincs_coverage.csv（is_landmark列を追加）')

# landmark遺伝子リストも保存
df_landmark_detail = pd.DataFrame({
    'gene': sorted(lincs_set),
    'is_landmark': [g in landmark_genes for g in sorted(lincs_set)],
    'in_gse92742': [g in gse92742_genes_clean for g in sorted(lincs_set)]
})
df_landmark_detail.to_csv(
    project_root / 'results' / 'data_qc' / 'glycogene_landmark_status.csv',
    index=False
)
print(f'保存: glycogene_landmark_status.csv')

## Supplementary Figure: Glycogene Coverage Visualization

Visualize the coverage relationships among GlycoEnzOnto, CycleGAN, GSE92742, and 978 Landmark genes using Venn diagrams, bar charts, and heatmaps.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from matplotlib.patches import Patch

# --- Figure style ---
plt.rcParams.update({
    'axes.labelsize': 18, 'axes.titlesize': 16,
    'xtick.labelsize': 16, 'ytick.labelsize': 16,
    'legend.fontsize': 16, 'font.size': 16,
    'svg.fonttype': 'none', 'axes.linewidth': 1.2,
})
C_BLUE = '#0077BB'
C_ORANGE = '#EE7733'
C_TEAL = '#009988'

# --- データ準備 ---
lincs_set = set(glyco_genes)  # CycleGAN 385
gse92742_clean = gse92742_genes - {'CELL', 'DOSE', 'PERTID', 'SAMPLE_ID', 'TIMEPOINT'}

# ========================================
# 2パネル構成: A=Venn, B=Stacked bar
# ========================================
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

# --- Panel A: 3-way Venn (GlycoEnzOnto / CycleGAN / GSE92742) ---
# Landmark情報をアノテーションで追加
v = venn3(
    [glycoenzonto_all, lincs_set, gse92742_clean],
    set_labels=('GlycoEnzOnto\n(403)', 'CycleGAN\n(385)', 'GSE92742\n(279)'),
    ax=ax_a
)
for patch in v.patches:
    if patch:
        patch.set_alpha(0.55)
for text in v.set_labels:
    if text:
        text.set_fontsize(14)
        text.set_fontweight('bold')
for text in v.subset_labels:
    if text:
        text.set_fontsize(14)
        text.set_fontweight('bold')

# Landmark annotation: CycleGAN 385中26個がlandmark
n_lm = len(lincs_set & landmark_genes)
n_inf = len(lincs_set) - n_lm
ax_a.annotate(
    f'CycleGAN 385 genes:\n  Landmark: {n_lm}  /  Inferred: {n_inf}',
    xy=(0.5, -0.12), xycoords='axes fraction', ha='center',
    fontsize=13, style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='gray', alpha=0.8)
)
ax_a.set_title('A', fontsize=18, fontweight='bold', loc='left', pad=10)

# --- Panel B: Stacked bar (Landmark vs Inferred per data source) ---
common_set = lincs_set & gse92742_clean
sources = ['CycleGAN\n(385)', f'GSE92742\n({len(gse92742_clean)})', f'Common\n({len(common_set)})']
landmark_counts = [
    len(lincs_set & landmark_genes),
    len(gse92742_clean & landmark_genes),
    len(common_set & landmark_genes)
]
inferred_counts = [
    len(lincs_set) - landmark_counts[0],
    len(gse92742_clean) - landmark_counts[1],
    len(common_set) - landmark_counts[2]
]

x = range(len(sources))
ax_b.bar(x, landmark_counts, color=C_BLUE, label='Landmark (directly measured)',
         edgecolor='white', linewidth=0.5)
ax_b.bar(x, inferred_counts, bottom=landmark_counts, color=C_ORANGE,
         label='Inferred (predicted)', edgecolor='white', linewidth=0.5)

for i in range(len(sources)):
    total = landmark_counts[i] + inferred_counts[i]
    if landmark_counts[i] > 0:
        ax_b.text(i, landmark_counts[i] / 2,
                  f'{landmark_counts[i]}\n({100 * landmark_counts[i] / total:.0f}%)',
                  ha='center', va='center', fontsize=13, fontweight='bold', color='white')
    if inferred_counts[i] > 0:
        ax_b.text(i, landmark_counts[i] + inferred_counts[i] / 2,
                  f'{inferred_counts[i]}\n({100 * inferred_counts[i] / total:.0f}%)',
                  ha='center', va='center', fontsize=13, fontweight='bold', color='white')

ax_b.set_xticks(x)
ax_b.set_xticklabels(sources)
ax_b.set_ylabel('Number of glycogenes')
ax_b.legend(loc='upper right')
ax_b.set_title('B', fontsize=18, fontweight='bold', loc='left', pad=10)
ax_b.grid(axis='y', alpha=0.3, linestyle='--')
for spine in ['top', 'right']:
    ax_b.spines[spine].set_visible(False)

plt.tight_layout()

# --- 保存 ---
suppl_dir = project_root / 'results' / 'data_qc'
suppl_dir.mkdir(parents=True, exist_ok=True)
for fmt in ['svg', 'png', 'pdf']:
    plt.savefig(suppl_dir / f'suppl_fig_glycogene_coverage.{fmt}',
                dpi=300, bbox_inches='tight', facecolor='white')
print(f'保存: {suppl_dir / "suppl_fig_glycogene_coverage.*"}')
plt.show()

In [ ]:
# パスウェイ別カバレッジ サマリ（論文記述用）
lincs_set = set(glyco_genes)
gse92742_clean = gse92742_genes - {'CELL', 'DOSE', 'PERTID', 'SAMPLE_ID', 'TIMEPOINT'}

pw_data = []
for pw_name, pw_genes in pathway_gene_map.items():
    pw_set = set(pw_genes)
    pw_data.append({
        'pathway': pw_name,
        'total': len(pw_set),
        'cyc': len(pw_set & lincs_set),
        'gse': len(pw_set & gse92742_clean),
        'cyc_miss': sorted(pw_set - lincs_set),
        'gse_miss': sorted(pw_set - gse92742_clean),
    })

df_pw = pd.DataFrame(pw_data)
df_pw['cyc%'] = (df_pw['cyc'] / df_pw['total'] * 100).round(1)
df_pw['gse%'] = (df_pw['gse'] / df_pw['total'] * 100).round(1)

# --- 全体サマリ ---
n_pw = len(df_pw)
cyc_full = (df_pw['cyc%'] == 100).sum()
gse_full = (df_pw['gse%'] == 100).sum()
cyc_median = df_pw['cyc%'].median()
gse_median = df_pw['gse%'].median()

print('=' * 70)
print('パスウェイ別カバレッジ サマリ（論文記述用）')
print('=' * 70)
print(f'GlycoEnzOntoパスウェイ数: {n_pw}')
print(f'GlycoEnzOnto全遺伝子数:  {len(glycoenzonto_all)}')
print()
print(f'              CycleGAN    GSE92742')
print(f'総遺伝子数:   {len(lincs_set):>5}       {len(gse92742_clean):>5}')
print(f'全体カバー率: {len(lincs_set & glycoenzonto_all)}/{len(glycoenzonto_all)} '
      f'({100*len(lincs_set & glycoenzonto_all)/len(glycoenzonto_all):.1f}%)  '
      f'{len(gse92742_clean & glycoenzonto_all)}/{len(glycoenzonto_all)} '
      f'({100*len(gse92742_clean & glycoenzonto_all)/len(glycoenzonto_all):.1f}%)')
print(f'カバー率中央値: {cyc_median:.1f}%      {gse_median:.1f}%')
print(f'100%カバーPW数: {cyc_full}/{n_pw}      {gse_full}/{n_pw}')
print()

# --- カバー率が低いパスウェイ（GSE92742で80%未満） ---
low_gse = df_pw[df_pw['gse%'] < 80].sort_values('gse%')
print(f'--- GSE92742カバー率 < 80% のパスウェイ ({len(low_gse)}件) ---')
for _, r in low_gse.iterrows():
    miss = '; '.join(r['gse_miss'])
    print(f'  {r["pathway"]}')
    print(f'    CycleGAN: {r["cyc"]}/{r["total"]} ({r["cyc%"]}%)  '
          f'GSE92742: {r["gse"]}/{r["total"]} ({r["gse%"]}%)')
    print(f'    GSE92742欠損: {miss}')
print()

# --- CycleGANでも100%でないパスウェイ ---
low_cyc = df_pw[df_pw['cyc%'] < 100].sort_values('cyc%')
print(f'--- CycleGANカバー率 < 100% のパスウェイ ({len(low_cyc)}件) ---')
for _, r in low_cyc.iterrows():
    miss = '; '.join(r['cyc_miss'])
    print(f'  {r["pathway"]}')
    print(f'    {r["cyc"]}/{r["total"]} ({r["cyc%"]}%)  欠損: {miss}')
print()

# --- 論文用の一文サマリ ---
print('=' * 70)
print('【論文用ドラフト文】')
print('=' * 70)
print(f'CycleGAN予測プロファイルはGlycoEnzOntoの{len(glycoenzonto_all)}遺伝子のうち'
      f'{len(lincs_set & glycoenzonto_all)}遺伝子'
      f'（{100*len(lincs_set & glycoenzonto_all)/len(glycoenzonto_all):.1f}%）をカバーし、'
      f'{n_pw}パスウェイ中{cyc_full}パスウェイで100%のカバレッジを達成した'
      f'（中央値{cyc_median:.1f}%）。'
      f'一方、オリジナルのGSE92742 L1000プロファイルでは'
      f'{len(gse92742_clean & glycoenzonto_all)}遺伝子'
      f'（{100*len(gse92742_clean & glycoenzonto_all)/len(glycoenzonto_all):.1f}%）のカバーにとどまり、'
      f'100%カバレッジのパスウェイは{n_pw}中{gse_full}'
      f'（中央値{gse_median:.1f}%）であった。'
      f'CycleGANデータセットで追加された{len(lincs_set - gse92742_clean)}遺伝子は'
      f'すべて推定値（非landmark）遺伝子であり、'
      f'特にドリコール前駆体生合成、GPIアンカー生合成、'
      f'UDP-グルクロン酸転移酵素（UGT）ファミリーのカバレッジを拡大した。')

## Extraction of Liver Cancer Cell Line Data

Extract data for target liver cancer cell lines from the GLYCO_GENES_WIDE table. Retrieve metadata (compound information, experimental conditions) and glycogene expression data.

In [ ]:
# Format glycogene columns for SQL
gene_columns = ', '.join([f'"{gene}"' for gene in glyco_genes])

# Convert cell line list to SQL string
cell_lines_str = "', '".join(cell_lines)

# Build query
query = f"""
SELECT 
    "sample_id",
    "canonical_smiles",
    "inchi_key",
    "pertname",
    "pertid",
    "cell",
    "dose",
    "timepoint",
    "cmapid",
    "compound_alias",
    {gene_columns}
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE
WHERE "cell" IN ('{cell_lines_str}')
"""

print("Executing query...")
df = pd.read_sql(query, conn)
print(f"Data retrieval complete: {len(df):,} records")
df.head()

## Verification of Data Completeness and z-score Distribution

Calculate QC statistics for the manuscript:
1. Missing value ratio for metadata
2. SD of median values per gene (indicator of systematic bias between genes)
3. SD of z-score distribution per sample (validity of normalization)

In [ ]:
# =============================================
# データ完全性とz-score分布の検証
# =============================================

meta_cols = ['sample_id', 'canonical_smiles', 'inchi_key', 'pertname',
             'pertid', 'cell', 'dose', 'timepoint', 'cmapid', 'compound_alias']
gene_cols = [c for c in df.columns if c not in meta_cols]

# glycogeneカラムを数値型に変換（Snowflakeから文字列として取得される場合がある）
df[gene_cols] = df[gene_cols].apply(pd.to_numeric, errors='coerce')

# --- 1. メタデータ欠損値 ---
required_meta = ['sample_id', 'pertname', 'pertid', 'cell', 'dose', 'timepoint']
meta_missing = df[required_meta].isnull().mean() * 100
print('=== 1. 必須メタデータの欠損値割合 ===')
for col in required_meta:
    print(f'  {col:20s}: {meta_missing[col]:.1f}%')
print(f'  → 必須メタデータ欠損率: {meta_missing.mean():.1f}%')

# --- 2. glycogene発現値の欠損値 ---
gene_missing = df[gene_cols].isnull().mean() * 100
print(f'\n=== 2. glycogene発現値の欠損値 ===')
print(f'  全エントリ数: {len(gene_cols)} genes × {len(df):,} samples = {len(gene_cols) * len(df):,}')
print(f'  欠損率（遺伝子ごと最大）: {gene_missing.max():.2f}%')
print(f'  欠損率（遺伝子ごと平均）: {gene_missing.mean():.4f}%')

# --- 3. 遺伝子ごとの中央値のSD ---
gene_medians = df[gene_cols].median()
sd_of_medians = gene_medians.std()
print(f'\n=== 3. 遺伝子ごとの中央値の標準偏差 ===')
print(f'  {len(gene_cols)} glycogene × {len(df):,}サンプルの遺伝子ごと中央値のSD: {sd_of_medians:.4f}')
print(f'  → 遺伝子間での系統的バイアスが小さいことを確認')

# --- 4. サンプルごとのz-score SD ---
sample_sds = df[gene_cols].std(axis=1)
print(f'\n=== 4. サンプルごとのz-score SD（{len(gene_cols)} glycogene横断） ===')
print(f'  平均SD:  {sample_sds.mean():.4f}')
print(f'  中央値SD: {sample_sds.median():.4f}')
print(f'  範囲:    {sample_sds.min():.4f} – {sample_sds.max():.4f}')

# --- サマリ（原稿用） ---
print(f'\n{"=" * 70}')
print('【原稿記載用サマリ】')
print(f'{"=" * 70}')
print(f'メタデータの欠損値は{meta_missing.mean():.0f}%であり、データの完全性が確認された。')
print(f'z-scoreで正規化された発現値について、{len(gene_cols)} glycogene × {len(df):,}サンプルの'
      f'全エントリにおける遺伝子ごとの中央値の標準偏差は≈{sd_of_medians:.4f}であり、'
      f'遺伝子間での系統的バイアスが小さいことが確認された'
      f'（個々のサンプル内でのz-score分布はSD≈{sample_sds.median():.0f}の標準的なスケールを維持している）。')

# =============================================
# Supplementary Figure: z-score QC
# =============================================
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.labelsize': 18, 'axes.titlesize': 16,
    'xtick.labelsize': 16, 'ytick.labelsize': 16,
    'legend.fontsize': 14, 'font.size': 16,
    'svg.fonttype': 'none', 'axes.linewidth': 1.2,
})
C_BLUE = '#0077BB'
C_ORANGE = '#EE7733'

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# --- Panel A: 遺伝子ごとの中央値の分布 ---
ax_a.hist(gene_medians.values, bins=50, color=C_BLUE, edgecolor='white', linewidth=0.5)
ax_a.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax_a.set_xlabel('Median z-score per gene')
ax_a.set_ylabel('Number of genes')
ax_a.set_title('A', fontsize=18, fontweight='bold', loc='left', pad=10)
ax_a.annotate(f'SD = {sd_of_medians:.4f}',
              xy=(0.95, 0.92), xycoords='axes fraction', ha='right',
              fontsize=14, bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0',
                                     edgecolor='gray', alpha=0.8))
for sp in ['top', 'right']:
    ax_a.spines[sp].set_visible(False)

# --- Panel B: サンプルごとのSD分布 ---
ax_b.hist(sample_sds.values, bins=50, color=C_ORANGE, edgecolor='white', linewidth=0.5)
ax_b.set_xlabel('SD of z-scores per sample\n(across 385 glycogenes)')
ax_b.set_ylabel('Number of samples')
ax_b.set_title('B', fontsize=18, fontweight='bold', loc='left', pad=10)
ax_b.annotate(f'Median = {sample_sds.median():.3f}',
              xy=(0.95, 0.92), xycoords='axes fraction', ha='right',
              fontsize=14, bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0',
                                     edgecolor='gray', alpha=0.8))
for sp in ['top', 'right']:
    ax_b.spines[sp].set_visible(False)

plt.tight_layout()

suppl_dir = project_root / 'results' / 'data_qc'
suppl_dir.mkdir(parents=True, exist_ok=True)
for fmt in ['svg', 'png', 'pdf']:
    plt.savefig(suppl_dir / f'suppl_fig_zscore_qc.{fmt}',
                dpi=300, bbox_inches='tight', facecolor='white')
print(f'保存: {suppl_dir / "suppl_fig_zscore_qc.*"}')
plt.show()

## Visualization Settings

Configure matplotlib and seaborn for publication-quality figures. Apply settings compatible with high-resolution output.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Publication-quality settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 13

# Results directory
results_dir = project_root / "results" / "qc"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Directory for saving figures: {results_dir}")

## GSE92742 Dataset Overview Summary and Comparison with CycleGAN

Retrieve similar statistics from the GSE92742 table and compare with the CycleGAN dataset.

In [ ]:
# GSE92742データの基本統計を取得
gse92742_stats_query = """
SELECT
    COUNT(*) AS total_experiments,
    COUNT(DISTINCT CELL) AS cell_lines,
    COUNT(DISTINCT PERTID) AS unique_compounds,
    COUNT(DISTINCT TIMEPOINT) AS timepoints,
    COUNT(DISTINCT DOSE) AS dose_levels
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE_GSE92742
WHERE CELL = 'HEPG2'
"""

gse92742_stats = pd.read_sql(gse92742_stats_query, conn).iloc[0]
print('GSE92742統計 (HEPG2):')
print(gse92742_stats)

# z-score統計（代表遺伝子1つで簡易取得）
sample_gene = sorted(gse92742_genes)[0]
zscore_query = f"""
SELECT AVG("{sample_gene}") AS mean_z, STDDEV("{sample_gene}") AS std_z
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE_GSE92742
WHERE CELL = 'HEPG2'
"""
try:
    gse92742_zscore = pd.read_sql(zscore_query, conn).iloc[0]
    gse92742_mean_z = f"{gse92742_zscore['MEAN_Z']:.4f}"
    gse92742_std_z = f"{gse92742_zscore['STD_Z']:.4f}"
except Exception as e:
    print(f'z-score統計取得エラー: {e}')
    gse92742_mean_z = 'N/A'
    gse92742_std_z = 'N/A'

# 比較テーブル作成
# CycleGANはpertname、GSE92742はPERTIDで化合物を数える
comparison_data = {
    'Metric': [
        'Total Experiments (HEPG2)',
        'Unique Compounds',
        'Glycogenes',
        'Timepoints',
        'Dose Levels',
        'Mean z-score',
        'Std z-score'
    ],
    'CycleGAN': [
        f"{len(df):,}",
        f"{df['pertname'].nunique():,}",
        f"{len(numeric_glyco_cols) if 'numeric_glyco_cols' in locals() else len(glyco_genes)}",
        f"{df['timepoint'].nunique()}",
        f"{df['dose'].nunique()}",
        f"{np.mean(sample_values):.4f}" if 'sample_values' in locals() else 'N/A',
        f"{np.std(sample_values):.4f}" if 'sample_values' in locals() else 'N/A'
    ],
    'GSE92742': [
        f"{int(gse92742_stats['TOTAL_EXPERIMENTS']):,}",
        f"{int(gse92742_stats['UNIQUE_COMPOUNDS']):,}",
        f"{len(gse92742_genes)}",
        f"{int(gse92742_stats['TIMEPOINTS'])}",
        f"{int(gse92742_stats['DOSE_LEVELS'])}",
        gse92742_mean_z,
        gse92742_std_z
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print('\n' + '=' * 80)
print('CycleGAN vs GSE92742 データセット概要比較 (HEPG2)')
print('=' * 80)
print(comparison_df.to_string(index=False))
print('=' * 80)
print(f'\n注: CycleGANの化合物数はpertname、GSE92742はPERTIDでカウント')
display(comparison_df)

# 比較テーブルを図として保存
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(
    cellText=comparison_df.values,
    colLabels=comparison_df.columns,
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
for i in range(len(comparison_df.columns)):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')
plt.title('CycleGAN vs GSE92742 Dataset Comparison (HEPG2)', fontweight='bold', fontsize=12, pad=20)

report_dir = project_root / 'results' / 'data_qc'
report_dir.mkdir(parents=True, exist_ok=True)
for fmt in ['png', 'pdf', 'svg']:
    plt.savefig(report_dir / f'dataset_comparison_cyclegan_vs_gse92742.{fmt}',
                bbox_inches='tight', facecolor='white', dpi=300)
comparison_df.to_csv(report_dir / 'dataset_comparison_cyclegan_vs_gse92742.csv', index=False)
print(f'\n保存: {report_dir / "dataset_comparison_cyclegan_vs_gse92742.*"}')
plt.show()

## Closing Snowflake Connection

Data extraction and visualization are complete, so close the Snowflake connection.

In [ ]:
conn.close()
print("Snowflake connection closed")